# Temporal Convolutional Network (TCN) for Solar Flare Classification

In [1]:
import os
import random
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sys, os
sys.path.append(os.path.abspath("../scripts"))

from data_utils import find_project_root, load_sequence_data
from preprocessing import compute_feature_stats, normalize_sequence_data
from metrics import compute_metrics, sweep_thresholds, format_metrics
from losses import FocalLoss

from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve, precision_recall_curve,
    confusion_matrix
)

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))
print('PyTorch:', torch.__version__)
print('Device:', DEVICE)

PyTorch: 2.11.0
Device: mps


In [ ]:
# 2) Load SWAN-SF (Train / Validation / Test)
SELECTED_FEATURES = ['SAVNCPP', 'ABSNJZH', 'R_VALUE', 'EPSZ', 'TOTFZ']

PROJECT_ROOT = find_project_root(Path(os.getcwd()))
DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'Cleaned SWANSF Dataset')

X_train, y_train, X_val, y_val, X_test, y_test = load_sequence_data(DATA_DIR, SELECTED_FEATURES)

print('Loaded data from:', DATA_DIR)
print('Train raw:', X_train.shape, y_train.shape)
print('Val raw:  ', X_val.shape, y_val.shape)
print('Test raw: ', X_test.shape, y_test.shape)

def print_pos_rate(y, name):
    y = np.asarray(y).astype(np.float32)
    print(f"{name} positive rate: {y.mean():.4f} ({int(y.sum())}/{len(y)})")

print_pos_rate(y_train, 'Train')
print_pos_rate(y_val, 'Validation')
print_pos_rate(y_test, 'Test')

Loaded data from: /Users/chintan/Drive/Projects/solar-flare-classification/data/Cleaned SWANSF Dataset
Train raw: (77865, 60, 5) (77865,)
Val raw:   (19899, 60, 5) (19899,)
Test raw:  (331185, 60, 5) (331185,)
Train positive rate: 0.4864 (37873/77865)
Validation positive rate: 0.4975 (9900/19899)
Test positive rate: 0.0188 (6234/331185)


In [ ]:
# 3) Prepare Arrays and Labels
SELECTED_FEATURES = ['SAVNCPP', 'ABSNJZH', 'R_VALUE', 'EPSZ', 'TOTFZ']

X_train = np.asarray(X_train).astype(np.float32)
X_val = np.asarray(X_val).astype(np.float32)
X_test = np.asarray(X_test).astype(np.float32)

y_train = np.asarray(y_train).astype(np.float32)
y_val = np.asarray(y_val).astype(np.float32)
y_test = np.asarray(y_test).astype(np.float32)

print('Selected features:', SELECTED_FEATURES)
print(f'Train X shape: {X_train.shape} | y shape: {y_train.shape} (1D labels) | y length: {len(y_train)}')
print(f'Val   X shape: {X_val.shape} | y shape: {y_val.shape} (1D labels) | y length: {len(y_val)}')
print(f'Test  X shape: {X_test.shape} | y shape: {y_test.shape} (1D labels) | y length: {len(y_test)}')

Selected features: ['SAVNCPP', 'ABSNJZH', 'R_VALUE', 'EPSZ', 'TOTFZ']
Train X shape: (77865, 60, 5) | y shape: (77865,) (1D labels) | y length: 77865
Val   X shape: (19899, 60, 5) | y shape: (19899,) (1D labels) | y length: 19899
Test  X shape: (331185, 60, 5) | y shape: (331185,) (1D labels) | y length: 331185


In [8]:
feature_mean, feature_std = compute_feature_stats(X_train)

X_train = normalize_sequence_data(X_train, feature_mean, feature_std)
X_val = normalize_sequence_data(X_val, feature_mean, feature_std)
X_test = normalize_sequence_data(X_test, feature_mean, feature_std)

print('Normalized train:', X_train.shape)
print('Normalized val:  ', X_val.shape)
print('Normalized test: ', X_test.shape)

Normalized train: (77865, 60, 5)
Normalized val:   (19899, 60, 5)
Normalized test:  (331185, 60, 5)


In [9]:
def show_class_stats(y, split_name):
    positive = int((y == 1).sum())
    total = len(y)
    negative = total - positive
    positive_rate = positive / total
    print(f'{split_name}: total={total}, pos={positive}, neg={negative}, pos_rate={positive_rate:.4f}')

show_class_stats(y_train, 'Train')
show_class_stats(y_val, 'Validation')
show_class_stats(y_test, 'Test')

majority_class = 1 if y_train.mean() >= 0.5 else 0
baseline_acc = float((y_test == majority_class).mean())
print(f'Majority-class baseline accuracy on test: {baseline_acc:.4f} (predict class {majority_class})')

Train: total=77865, pos=37873, neg=39992, pos_rate=0.4864
Validation: total=19899, pos=9900, neg=9999, pos_rate=0.4975
Test: total=331185, pos=6234, neg=324951, pos_rate=0.0188
Majority-class baseline accuracy on test: 0.9812 (predict class 0)


In [10]:
X_train_t = torch.from_numpy(X_train).float()
X_val_t = torch.from_numpy(X_val).float()
X_test_t = torch.from_numpy(X_test).float()

y_train_t = torch.from_numpy(y_train).float()
y_val_t = torch.from_numpy(y_val).float()
y_test_t = torch.from_numpy(y_test).float()

train_ds = TensorDataset(X_train_t, y_train_t)
val_ds = TensorDataset(X_val_t, y_val_t)
test_ds = TensorDataset(X_test_t, y_test_t)

BATCH_SIZE = 256
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print('DataLoaders ready.')

DataLoaders ready.


In [ ]:
#Loss Function (BCE or Focal)
num_pos = float((y_train == 1).sum())
num_neg = float((y_train == 0).sum())
pos_weight_value = num_neg / max(num_pos, 1.0)

# Choose ONE loss (uncomment what you want)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight_value, dtype=torch.float32, device=DEVICE))  # BCE baseline
#criterion = FocalLoss(alpha=0.75, gamma=2.0)  # Focal Loss

print(f'Class imbalance ratio (neg/pos) = {pos_weight_value:.4f}')

Class imbalance ratio (neg/pos) = 1.0560


In [20]:
class TCNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation, dropout):
        super().__init__()
        padding = (kernel_size - 1) * dilation

        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, dilation=dilation, padding=padding)
        self.relu1 = nn.ReLU()
        self.drop1 = nn.Dropout(dropout)

        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, dilation=dilation, padding=padding)
        self.relu2 = nn.ReLU()
        self.drop2 = nn.Dropout(dropout)

        self.residual_proj = nn.Conv1d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else None

    def chomp(self, x, padding):
        return x[:, :, :-padding] if padding > 0 else x

    def forward(self, x):
        p1 = self.conv1.padding[0]
        out = self.conv1(x)
        out = self.chomp(out, p1)
        out = self.relu1(out)
        out = self.drop1(out)

        p2 = self.conv2.padding[0]
        out = self.conv2(out)
        out = self.chomp(out, p2)
        out = self.relu2(out)
        out = self.drop2(out)

        residual = x if self.residual_proj is None else self.residual_proj(x)
        return out + residual

class SimpleTCN(nn.Module):
    def __init__(self, input_features, channels=64, kernel_size=3, dilations=(1, 2, 4, 8), dropout=0.2):
        super().__init__()
        blocks = []
        in_ch = input_features
        # Dilation expands temporal receptive field without deepening the network too much.
        for d in dilations:
            blocks.append(TCNBlock(in_ch, channels, kernel_size, d, dropout))
            in_ch = channels
        self.tcn = nn.Sequential(*blocks)
        self.classifier = nn.Linear(channels, 1)

    def forward(self, x):
        # x: (batch, timesteps, features)
        x = x.transpose(1, 2)  # -> (batch, features, timesteps)
        z = self.tcn(x)
        # Global average pooling summarizes the whole sequence into one vector per sample.
        z = z.mean(dim=2)
        logits = self.classifier(z).squeeze(1)
        return logits

model = SimpleTCN(input_features=X_train.shape[2], channels=64, kernel_size=3, dilations=(1, 2, 4, 8), dropout=0.2).to(DEVICE)
print(model)

SimpleTCN(
  (tcn): Sequential(
    (0): TCNBlock(
      (conv1): Conv1d(5, 64, kernel_size=(3,), stride=(1,), padding=(2,))
      (relu1): ReLU()
      (drop1): Dropout(p=0.2, inplace=False)
      (conv2): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(2,))
      (relu2): ReLU()
      (drop2): Dropout(p=0.2, inplace=False)
      (residual_proj): Conv1d(5, 64, kernel_size=(1,), stride=(1,))
    )
    (1): TCNBlock(
      (conv1): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(4,), dilation=(2,))
      (relu1): ReLU()
      (drop1): Dropout(p=0.2, inplace=False)
      (conv2): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(4,), dilation=(2,))
      (relu2): ReLU()
      (drop2): Dropout(p=0.2, inplace=False)
    )
    (2): TCNBlock(
      (conv1): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(8,), dilation=(4,))
      (relu1): ReLU()
      (drop1): Dropout(p=0.2, inplace=False)
      (conv2): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(8,), di

In [21]:
THRESHOLD_GRID = np.linspace(0.1, 0.9, 33)
MAX_ACCEPTABLE_FPR = 0.10

print('THRESHOLD_GRID size:', len(THRESHOLD_GRID))
print('MAX_ACCEPTABLE_FPR:', MAX_ACCEPTABLE_FPR)

THRESHOLD_GRID size: 33
MAX_ACCEPTABLE_FPR: 0.1


In [22]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=2
)

def run_epoch(model, loader, optimizer=None):
    train_mode = optimizer is not None
    model.train() if train_mode else model.eval()

    total_loss = 0.0
    total_count = 0

    for xb, yb in loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        if train_mode:
            optimizer.zero_grad()

        with torch.set_grad_enabled(train_mode):
            logits = model(xb)
            loss = criterion(logits, yb)
            if train_mode:
                loss.backward()
                # Prevents exploding gradients and stabilizes training
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()

        batch_size = xb.size(0)
        total_loss += loss.item() * batch_size
        total_count += batch_size

    return total_loss / max(total_count, 1)

def predict_probabilities_from_loader(model, loader):
    probs_all = []
    model.eval()
    with torch.no_grad():
        for xb, _ in loader:
            xb = xb.to(DEVICE)
            logits = model(xb)
            probs_all.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(probs_all).ravel()

EPOCHS = 25
PATIENCE = 5

history = {'train_loss': [], 'val_tss': [], 'lr': []}
best_val_tss = -1e9
best_state = None
bad_epochs = 0

for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(model, train_loader, optimizer=optimizer)

    y_val_prob_epoch = predict_probabilities_from_loader(model, val_loader)
    val_pick, _ = sweep_thresholds(
        y_val.astype(int),
        y_val_prob_epoch,
        thresholds=THRESHOLD_GRID,
        max_fpr=MAX_ACCEPTABLE_FPR,
    )
    val_tss = val_pick['tss']

    scheduler.step(val_tss)
    lr = optimizer.param_groups[0]['lr']

    history['train_loss'].append(train_loss)
    history['val_tss'].append(val_tss)
    history['lr'].append(lr)

    print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_tss={val_tss:.4f} | lr={lr:.6f}")

    if val_tss > best_val_tss:
        best_val_tss = val_tss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        bad_epochs = 0
    else:
        bad_epochs += 1

    if bad_epochs >= PATIENCE:
        print(f'Early stopping at epoch {epoch}')
        break

if best_state is not None:
    model.load_state_dict(best_state)
    model.to(DEVICE)
    print(f'Loaded best model (val_tss={best_val_tss:.4f})')

Epoch 01 | train_loss=0.1348 | val_tss=0.8998 | lr=0.001000
Epoch 02 | train_loss=0.1050 | val_tss=0.9088 | lr=0.001000
Epoch 03 | train_loss=0.0925 | val_tss=0.8892 | lr=0.001000
Epoch 04 | train_loss=0.0810 | val_tss=0.9018 | lr=0.001000
Epoch 05 | train_loss=0.0727 | val_tss=0.9147 | lr=0.001000
Epoch 06 | train_loss=0.0659 | val_tss=0.9136 | lr=0.001000
Epoch 07 | train_loss=0.0605 | val_tss=0.9179 | lr=0.001000
Epoch 08 | train_loss=0.0567 | val_tss=0.8983 | lr=0.001000
Epoch 09 | train_loss=0.0513 | val_tss=0.9032 | lr=0.001000
Epoch 10 | train_loss=0.0487 | val_tss=0.8884 | lr=0.000500
Epoch 11 | train_loss=0.0394 | val_tss=0.8959 | lr=0.000500
Epoch 12 | train_loss=0.0370 | val_tss=0.8580 | lr=0.000500
Early stopping at epoch 12
Loaded best model (val_tss=0.9179)


In [23]:
val_probs = predict_probabilities_from_loader(model, val_loader)
test_probs = predict_probabilities_from_loader(model, test_loader)
y_test_prob = test_probs  # alias used by downstream plotting cell

val_best, _ = sweep_thresholds(
    y_val.astype(int),
    val_probs,
    thresholds=THRESHOLD_GRID,
    max_fpr=MAX_ACCEPTABLE_FPR,
 )
best_threshold = val_best['threshold']

test_metrics = compute_metrics(
    y_test.astype(int),
    test_probs,
    threshold=val_best['threshold']
 )

test_constrained, _ = sweep_thresholds(
    y_test.astype(int),
    test_probs,
    thresholds=THRESHOLD_GRID,
    max_fpr=MAX_ACCEPTABLE_FPR
 )

rows = [
    {
        'Split': 'Validation (best under FPR constraint)',
        'Threshold': val_best['threshold'],
        'ROC-AUC': val_best['roc_auc'],
        'PR-AUC': val_best['pr_auc'],
        'TPR': val_best['tpr'],
        'FPR': val_best['fpr'],
        'TSS': val_best['tss'],
        'HSS': val_best['hss']
    },
    {
        'Split': 'Test (validation-calibrated threshold)',
        'Threshold': test_metrics['threshold'],
        'ROC-AUC': test_metrics['roc_auc'],
        'PR-AUC': test_metrics['pr_auc'],
        'TPR': test_metrics['tpr'],
        'FPR': test_metrics['fpr'],
        'TSS': test_metrics['tss'],
        'HSS': test_metrics['hss']
    },
    {
        'Split': 'Test (oracle threshold, upper bound)',
        'Threshold': test_constrained['threshold'],
        'ROC-AUC': test_constrained['roc_auc'],
        'PR-AUC': test_constrained['pr_auc'],
        'TPR': test_constrained['tpr'],
        'FPR': test_constrained['fpr'],
        'TSS': test_constrained['tss'],
        'HSS': test_constrained['hss']
    },
]

results_df = pd.DataFrame(rows, columns=[
    'Split', 'Threshold', 'ROC-AUC', 'PR-AUC', 'TPR', 'FPR', 'TSS', 'HSS'
])
display(results_df.style.format({
    'Threshold': '{:.4f}', 'ROC-AUC': '{:.4f}', 'PR-AUC': '{:.4f}',
    'TPR': '{:.4f}', 'FPR': '{:.4f}', 'TSS': '{:.4f}', 'HSS': '{:.4f}'
}))

,Split,Threshold,ROC-AUC,PR-AUC,TPR,FPR,TSS,HSS
0,Validation (best under FPR constraint),0.1500,0.9911,0.9904,0.9543,0.0364,0.9179,0.9180
1,Test (validation-calibrated threshold),0.1500,0.9000,0.3158,0.6866,0.0763,0.6102,0.2182
2,"Test (oracle threshold, upper bound)",0.1000,0.9000,0.3158,0.7007,0.0839,0.6168,0.2058
